# Práctica 3. Grafos de conocimiento
## Ingeniería del Conocimiento    2025/2026
### Prof. Juan A. Recio García

# Tutorial rápido SPARQL para Consultas en Wikidata

python version: 3.12.7

## 1. Estructura Básica de una Consulta

```sparql
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?variable1 ?variable2
WHERE {
    # Patrones de búsqueda
}
LIMIT 50
```

### Componentes esenciales:
- **PREFIX**: Define abreviaturas para URIs largos
- **SELECT**: Especifica qué variables queremos en los resultados
- **WHERE**: Contiene los patrones de búsqueda (tripletas)
- **LIMIT**: Limita el número de resultados

## 2. Tripletas Básicas

Las tripletas siguen el patrón: `sujeto predicado objeto`

```sparql
?pelicula wdt:P31 wd:Q11424 .     # ?pelicula es una instancia de película
?pelicula wdt:P57 wd:Q25191 .     # ?pelicula tiene como director a Nolan
?pelicula rdfs:label ?titulo .    # obtiene el título de la película
```

Las propiedades (predicado) tienen el prefijo `wdt`. Las entidades el prefijo `wd`.

Puedes buscar los códigos en la página de wikidata: https://www.wikidata.org/

## 3. Obtener Etiquetas Legibles

Las entidades son objetos con distintas propiedades. Para obtener la etiqueta legible usa `rdfs:label`.

```sparql
?entidad rdfs:label ?nombre .
FILTER (lang(?nombre) = 'es')     # Solo en español
FILTER (lang(?nombre) = 'en')     # Solo en inglés
```

## 4. OPTIONAL - Datos Opcionales

Cuando un dato puede no existir para todas las entidades:

```sparql
OPTIONAL { ?pelicula wdt:P2047 ?duracion }
OPTIONAL { ?pelicula wdt:P2130 ?presupuesto }
```

Es muy útil porque normalmente faltan datos y si no se utiliza obtenemos consultas vacías.

## 5. BIND - Crear Nuevas Variables

```sparql
?pelicula wdt:P577 ?fecha .
BIND(YEAR(?fecha) AS ?año)        # Extrae solo el año de una fecha
```

## 6. FILTER - Filtrar Resultados

### Filtros de idioma:
```sparql
FILTER (lang(?titulo) = 'en')
```

Si no se utiliza se repetirán filas con el mismo resultado pero con la etiqueta correspondiente en todos los idiomas disponibles.

### Filtros numéricos:
```sparql
FILTER (?año >= 2015)
FILTER (?altura > 50)
FILTER (?altura > 1.70 && ?altura < 2.00)
FILTER (?año < 1700)
```

### Verificar si una variable tiene valor:
```sparql
FILTER (BOUND(?duracion))
FILTER (BOUND(?movimiento))
```
Obligamos a que no haya resultados sin ese valor.

## 7. Navegación por Jerarquías

Para incluir subclases en la búsqueda:

```sparql
?libro wdt:P31/wdt:P279* wd:Q7725634    # instancia de o subclase de obra literaria
?museo wdt:P31/wdt:P279* wd:Q33506       # instancia de o subclase de museo
```

Si solo se utiliza `wdt:P31` (instance of) solo se devuelven las instancias directas de la clase.

## 8. DISTINCT - Eliminar Duplicados

```sparql
SELECT DISTINCT ?pelicula ?titulo ?año
```

## 9. GROUP_CONCAT - Concatenar Múltiples Valores

Cuando una entidad tiene varios valores para una propiedad:

```sparql
SELECT ?titulo (GROUP_CONCAT(DISTINCT ?ingrediente; separator=", ") AS ?ingredientes)
WHERE {
    # consulta para ?ingrediente
}
GROUP BY ?titulo
```

Así evitamos que se dupliquen filas con los valores de esa propiedad.

## 10. Agregaciones

### Contar elementos:
```sparql
SELECT ?equipo (COUNT(?titulo) AS ?titulos){
    # consulta para titulo
}
GROUP BY ?equipo
```

### Obtener el máximo:
```sparql
SELECT (MAX(?visitantes) AS ?maxVisitantes) {
    # consulta para visitantes
}
GROUP BY ?museo
```

## 11. ORDER BY - Ordenar Resultados

Se indica al final de la consulta: 

```sparql
ORDER BY DESC(?año)        # Descendente
ORDER BY ASC(?año)         # Ascendente
```



## 14. Consejos Prácticos

1. **Siempre usa DISTINCT** si sospechas que puede haber duplicados
2. **OPTIONAL evita consultas vacías y fallos** cuando no todas las entidades tienen todos los datos.
3. **Filtra el idioma** para obtener nombres legibles y evitar filas duplicadas con propiedades en varios idiomas
4. **Usa LIMIT** para no saturar el servidor mientras pruebas
5. **GROUP BY** es necesario cuando usas GROUP_CONCAT o funciones de agregación
6. **Termina cada tripleta con punto (.)** excepto la última del bloque
7. **Usa punto y coma (;)** para continuar agregando predicados al mismo sujeto
8. **Los comentarios** se escriben con #

## 15. Patrones Comunes

### Obtener entidades con sus nombres:
```sparql
?entidad wdt:P31 wd:QXXXXX ;
         rdfs:label ?nombre .
FILTER (lang(?nombre) = 'en')
```

### Navegar relaciones:
```sparql
?entidad wdt:P123 ?editorialEntity .
?editorialEntity rdfs:label ?editorial .
FILTER (lang(?editorial) = 'en')
```

### Fechas y años:
```sparql
?entidad wdt:P577 ?fecha .
BIND(YEAR(?fecha) AS ?año)
FILTER (?año >= 2000 && ?año <= 2024)
```

### Múltiples valores concatenados:
```sparql
SELECT ?entidad (GROUP_CONCAT(DISTINCT ?valor; separator=", ") AS ?valores)
WHERE {
    ?entidad wdt:PXXXX ?valorEntity .
    ?valorEntity rdfs:label ?valor .
    FILTER (lang(?valor) = 'en')
}
GROUP BY ?entidad
```

# Practica consultas SPARQL con WIKIDATA

Utilizar la página de consultas de WIKIDATA para ejecutar las siguientes consultas SPAQL y averiguar cómo conseguir las que se dejan como ejercicio.

In [1]:
# EJECUTA ESTA CELDA PARA ABRIR LA PÁGINA DE CONSULTAS DE WIKIDATA


import webbrowser
import urllib

# Utiliza esta dirección para abrir la página de consultas de Wikidata con una consulta SPARQL predefinida. 
main_url = "https://query.wikidata.org/"

# Función para abrir la URL con la consulta SPARQL
def open_query(query=""):
    url = main_url + "#" + urllib.parse.quote_plus(query).replace("+", "%20")
    webbrowser.open(url, new=0, autoraise=True)


open_query()


## Dominio 1: Películas y series

### Consulta 1.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener películas de Chistopher Nolan y año de estreno

In [2]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?pelicula ?titulo ?año
WHERE {
    ?pelicula wdt:P31 wd:Q11424 ;         # instancia de: película
            wdt:P57 wd:Q25191 ;          # director: Christopher Nolan (Q25191)
            rdfs:label ?titulo ;
            wdt:P577 ?fecha .            # fecha de publicación
    
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?titulo) = 'en')
}
ORDER BY DESC(?año)
"""
open_query(query)

### Consulta 1.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener series de televisión con su creador y año de inicio

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?serie ?titulo ?creador ?año
WHERE {
    ?serie wdt:P31 wd:Q5398426 ;          # instancia de: serie de televisión
           rdfs:label ?titulo ;
           wdt:P170 ?creadorEntity ;       # creador
           wdt:P571 ?fecha .               # fecha de inicio
           ?creadorEntity rdfs:label ?creador .

    BIND(YEAR(?fecha) AS ?año)

    FILTER (lang(?titulo) = 'es')
    FILTER (lang(?creador) = 'es')
"""
open_query(query)

## Dominio 2: Música

### Consulta 2.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener canciones de Michael Jackson

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?cancion ?titulo
WHERE {
    ?cancion wdt:P31 wd:Q7366 ;           # instancia de: canción
            wdt:P175 wd:Q2831 ;           # intérprete: Michael Jackson
            rdfs:label ?titulo .
    
    FILTER (lang(?titulo) = 'en')
}
LIMIT 50
"""
open_query(query)

### Consulta 2.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener álbumes de "The Beatles" con fecha de publicación


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?album ?titulo ?año
WHERE {
    ?album wdt:P31 wd:Q482994 ;           # instancia de: álbum musical
           wdt:P175 wd:Q1299 ;            # intérprete: The Beatles (Q1299)
           rdfs:label ?titulo ;
           wdt:P577 ?fecha .              # fecha de publicación

    BIND(YEAR(?fecha) AS ?año)

    FILTER (lang(?titulo) = 'en')
}
ORDER BY ASC(?año)

"""
open_query(query)

## Dominio 3: Recetas y gastronomía


### Consulta 3.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener platos típicos de España

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?plato ?nombre
WHERE {
    ?plato wdt:P31 wd:Q746549 ; # instancia de: plato
           wdt:P495 wd:Q29;
           rdfs:label ?nombre.
    
    
    FILTER (lang(?nombre) = 'es')
}
LIMIT 50
"""
open_query(query)

### Consulta 3.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener bebidas con su país de origen

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?bebida ?nombre ?pais
WHERE {
    ?bebida wdt:P31/wdt:P279* wd:Q40050 ;  # instancia de: bebida (Q40050)
            rdfs:label ?nombre ;
            wdt:P495 ?paisEntity .          # país de origen

    ?paisEntity rdfs:label ?pais .

    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?pais) = 'es')
}
LIMIT 50

"""
open_query(query)

## Dominio 4: Productos tecnológicos


### Consulta 4.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener smartphones con su año de lanzamiento


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?telefono ?nombre ?año
WHERE {
    ?telefono wdt:P31 wd:Q19723451 ;         # instancia de: smartphone
              rdfs:label ?nombre ;
              wdt:P571 ?fecha .
        
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?nombre) = 'en')
}
ORDER BY DESC(?año)
"""
open_query(query)

### Consulta 4.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener teléfonos con su fabricante y año de lanzamiento


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?telefono ?nombre ?fabricante ?año
WHERE {
    ?telefono wdt:P31 wd:Q19723451 ;       # instancia de: smartphone
              rdfs:label ?nombre ;
              wdt:P176 ?fabricanteEntity ;  # fabricante
              wdt:P571 ?fecha .             # fecha de lanzamiento

    ?fabricanteEntity rdfs:label ?fabricante .

    BIND(YEAR(?fecha) AS ?año)

    FILTER (lang(?nombre) = 'en')
    FILTER (lang(?fabricante) = 'en')
}
ORDER BY DESC(?año)

"""
open_query(query)

## Dominio 5: Turismo y viajes


### Consulta 5.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener monumentos de España con su ciudad


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?monumento ?nombre ?ciudad
WHERE {
    ?monumento wdt:P31/wdt:P279*  wd:Q4989906 ;      # instancia de: monumento
               rdfs:label ?nombre ;
               wdt:P17 wd:Q29;
               wdt:P131 ?ciudadEntity .   # ubicado en
    
    ?ciudadEntity rdfs:label ?ciudad.
      
    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?ciudad) = 'es')
}
"""
open_query(query)

### Consulta 5.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener museos de historia de la Unión Europea con su ciudad y país


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?museo ?nombre ?ciudad ?pais
WHERE {
    ?museo wdt:P31/wdt:P279* wd:Q200764 ;  # instancia de: museo de historia
           rdfs:label ?nombre ;
           wdt:P131 ?ciudadEntity ;         # ubicado en
           wdt:P17 ?paisEntity .            # país

    # El país debe ser miembro de la UE
    ?paisEntity wdt:P463 wd:Q458 .          # miembro de: Unión Europea (Q458)

    ?ciudadEntity rdfs:label ?ciudad .
    ?paisEntity rdfs:label ?pais .

    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?ciudad) = 'es')
    FILTER (lang(?pais) = 'es')
}
LIMIT 50

"""
open_query(query)

## Dominio 6: Inmobiliario


### Consulta 6.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener edificios situados en las calles conectadas a la Calle Alcalá (Madrid)


In [4]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?calles ?edificio
WHERE {
    ?callesEntity wdt:P2789 wd:Q2424746.        # instancia de: edificio
    ?callesEntity rdfs:label ?calles.
    
    ?edificioEntity wdt:P669 ?callesEntity.
    ?edificioEntity rdfs:label ?edificio.
    FILTER (lang(?calles) = 'es')
    FILTER (lang(?edificio) = 'es')

}
"""
open_query(query)

### Consulta 6.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener edificios situados en las calles conectadas a la Calle Alcalá (Madrid) que estén declarados bien de interes cultural y fueron creados antes de 1900.

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?calle ?edificio ?año
WHERE {
    ?calleEntity wdt:P2789 wd:Q2424746 .    # calles conectadas a Calle Alcalá
    ?calleEntity rdfs:label ?calle .

    ?edificioEntity wdt:P669 ?calleEntity ; # ubicado en la calle
                    rdfs:label ?edificio ;
                    wdt:P31/wdt:P279* wd:Q23712 ;  # bien de interés cultural
                    wdt:P571 ?fecha .               # fecha de construcción

    BIND(YEAR(?fecha) AS ?año)
    FILTER (?año < 1900)

    FILTER (lang(?calle) = 'es')
    FILTER (lang(?edificio) = 'es')
}
ORDER BY ASC(?año)

"""
open_query(query)

## Dominio 7: Libros y literatura


### Consulta 7.1 - Nivel Básico (Ejemplo)
**Objetivo**: Series de manga creadas por Akira Toriyama, junto con número de partes que las componen y publisher.

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?libro ?titulo ?partes ?publisher
WHERE {
    ?libro wdt:P50 wd:Q208582 ;          # instancia de: obra literaria
           wdt:P31 wd:Q21198342;
           rdfs:label ?titulo .
  
    OPTIONAL {?libro wdt:P2635 ?partes.} 
    OPTIONAL {?libro wdt:P123 ?publisherEntity. 
             ?publisherEntity rdfs:label ?publisher.
             FILTER (lang(?publisher) = 'en')
    }
   
    FILTER (lang(?titulo) = 'en')
}
"""
open_query(query)

### Consulta 7.2 - Nivel Básico (Ejercicio)
**Objetivo**: Comics Españoles publicados antes de 1990, con título, año y creadores (concatenados si hay varios)

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?comic ?titulo ?año
    (GROUP_CONCAT(DISTINCT ?creador; separator=", ") AS ?creadores)
WHERE {
    ?comic wdt:P31/wdt:P279* wd:Q1004 ;    # instancia de: cómic
           wdt:P495 wd:Q29 ;               # país de origen: España
           rdfs:label ?titulo ;
           wdt:P577 ?fecha .               # fecha de publicación

    OPTIONAL {
        ?comic wdt:P50 ?creadorEntity .    # autor
        ?creadorEntity rdfs:label ?creador .
        FILTER (lang(?creador) = 'es')
    }

    BIND(YEAR(?fecha) AS ?año)
    FILTER (?año < 1990)
    FILTER (lang(?titulo) = 'es')
}
GROUP BY ?comic ?titulo ?año
ORDER BY ASC(?año)

"""
open_query(query)

## Dominio 8: Salud y síntomas médicos


### Consulta 8.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener enfermedades cuya causa sea fumar, con sus síntomas principales agrupadas

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?enfermedad ?nombre (GROUP_CONCAT(DISTINCT ?sintoma; separator=", ") AS ?sintomas) 
WHERE {
    ?enfermedad wdt:P31/wdt:P279* wd:Q12136 ;       # instancia de: enfermedad
                wdt:P828 wd:Q662860;
                rdfs:label ?nombre .
    
    OPTIONAL { ?enfermedad wdt:P780 ?sintomaEntity .
               ?sintomaEntity rdfs:label ?sintoma .
               FILTER (lang(?sintoma) = 'en')
    }
    
    FILTER (lang(?nombre) = 'en')
}
GROUP BY ?enfermedad ?nombre
LIMIT 50
"""
open_query(query)

### Consulta 8.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener medicamentos (Límite 50) con sus usos terapéuticos


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?medicamento ?nombre
    (GROUP_CONCAT(DISTINCT ?uso; separator=", ") AS ?usosTerapeuticos)
WHERE {
    ?medicamento wdt:P31/wdt:P279* wd:Q12140 ;  # instancia de: medicamento
                 rdfs:label ?nombre .

    OPTIONAL {
        ?medicamento wdt:P2175 ?usoEntity .      # condición médica tratada
        ?usoEntity rdfs:label ?uso .
        FILTER (lang(?uso) = 'en')
    }

    FILTER (lang(?nombre) = 'en')
}
GROUP BY ?medicamento ?nombre
LIMIT 50

"""
open_query(query)

## Dominio 9: Deportes (Fútbol)

### Consulta 9.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener futbolistas españoles con su posición y equipo actual


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?jugador ?nombre ?posicion ?equipo
WHERE {
    ?jugador wdt:P31 wd:Q5 ;              # instancia de: humano
             wdt:P106 wd:Q937857 ;         # ocupación: futbolista
             rdfs:label ?nombre ;
             wdt:P413 ?posicionEntity ;
             wdt:P54 ?equipoEntity .

    ?jugador wdt:P27 wd:Q29.
    ?posicionEntity rdfs:label ?posicion .
    ?equipoEntity rdfs:label ?equipo .
    
    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?posicion) = 'es')
    FILTER (lang(?equipo) = 'es')
}
LIMIT 50
"""
open_query(query)

### Consulta 9.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener equipos españoles de la liga


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?equipo ?nombre ?ciudad
WHERE {
    ?equipo wdt:P31 wd:Q476028 ;           # instancia de: club de fútbol
            wdt:P118 wd:Q324867 ;          # liga: La Liga (Q324867)
            rdfs:label ?nombre ;
            wdt:P131 ?ciudadEntity .        # ciudad sede

    ?ciudadEntity rdfs:label ?ciudad .

    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?ciudad) = 'es')
}
ORDER BY ASC(?nombre)

"""
open_query(query)

## Dominio 10: Arte y museos


### Consulta 10.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener pinturas españolas con su autor y año de creación


In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pintura ?titulo ?artista ?año
WHERE {
    ?pintura wdt:P31 wd:Q3305213 ;        # instancia de: pintura
             wdt:P17 wd:Q29;
             rdfs:label ?titulo ;
             wdt:P170 ?artistaEntity ;
             wdt:P571 ?fecha .
    
    ?artistaEntity rdfs:label ?artista .
    
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?titulo) = 'en')
    FILTER (lang(?artista) = 'en')
}
ORDER BY DESC(?año)
LIMIT 50
"""
open_query(query)

### Consulta 10.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener pinturas del Museo del Prado anteriores a 1700 con su autor y año de creación

In [ ]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?pintura ?titulo ?artista ?año
WHERE {
    ?pintura wdt:P31 wd:Q3305213 ;         # instancia de: pintura
             wdt:P276 wd:Q160112 ;         # ubicada en: Museo del Prado (Q160112)
             rdfs:label ?titulo ;
             wdt:P170 ?artistaEntity ;      # creador/artista
             wdt:P571 ?fecha .              # fecha de creación

    ?artistaEntity rdfs:label ?artista .

    BIND(YEAR(?fecha) AS ?año)
    FILTER (?año < 1700)

    FILTER (lang(?titulo) = 'es')
    FILTER (lang(?artista) = 'es')
}
ORDER BY ASC(?año)
LIMIT 100

"""
open_query(query)